# 14 - Composition des formes quadratiques binaires et nombre de classes h(D)
*A Course in Arithmetic*, chapitre 4 (Serre, GTM 7) — *Disquisitiones Arithmeticae* §§ 222-256 (Gauss, 1801).
Le carnet 11 a établi la densité de Chebotarev 1/2 et la liste des 9 corps quadratiques imaginaires de Heegner pour lesquels h(D) = 1. Le carnet 13 a posé le symbole de Jacobi et l'extension Kronecker en O(log n). Ce carnet 14 franchit une étape : **comment calculer h(D)** ? La réponse canonique pour les **discriminants quadratiques imaginaires** D < 2 est l'algorithme de **réduction de Gauss** sur les formes quadratiques binaires — l'ancêtre de l'arithmétique des corps de nombres, antérieur de 100 ans aux idéaux de Dedekind.
Outils : Python stdlib pur (`math.isqrt`). Le carnet 11 a posé la notion de h(D) (citation en fin de parcours) ; ce 14 la calcule et la vérifie numériquement.
Puis il franchit le pas suivant : l'ensemble des classes n'est pas un simple compteur — Gauss l'a muni d'une **multiplication**, la *composition*, qui en fait un **groupe abélien fini d'ordre h(D)**. La section 5 construit cette loi et en vérifie les axiomes un par un.

## Plan
1. Formes quadratiques binaires (a, b, c) de discriminant D = b² - 4ac, et réduction de Gauss (pas `b → b ± 2a`, `swap(a,c)`)
2. Nombre de classes h(D) pour D < 0 : énumération des formes réduites
3. Vérification empirique du théorème de Stark-Heegner sur les 9 corps de Heegner (h(D) = 1)
4. Exemples guidés : balayage de h(D) et formule analytique de Dirichlet
5. **Composition de Gauss** : le produit de deux classes et la loi de groupe (fermeture, neutre, inverse, commutativité, associativité)
6. Exercices


## 1. Formes quadratiques binaires
Une **forme quadratique binaire** de discriminant D est un triplet (a, b, c) avec a > 0 et
$$f(x, y) = ax^2 + bxy + cy^2, \qquad D = b^2 - 4ac.$$
Pour un **corps quadratique** $K = \mathbb{Q}(\sqrt{D})$ de discriminant quadratique fondamental D (premier ou produit d'impar d ≥ 1), chaque idéal de $\mathcal{O}_K$ correspond à une classe de formes équivalentes sous l'action de SL₂(Z). Le **nombre de classes** h(D) est le nombre de classes d'équivalence de formes **réduites** :
- $|b| \leq a \leq c$ ;
- si $|b| = a$ ou $a = c$, alors $b \geq 0$.
Pour D < 0 (corps quadratique imaginaire), h(D) est **fini**. Pour D > 1 (corps quadratique réel), h(D) est **fini également** — c'est le **groupe des unités** qui est infini (±ε^k pour l'unité fondamentale ε, théorème des unités de Dirichlet) ; les formes réduites indéfinies s'y organisent en cycles, un par classe.

In [1]:
from math import isqrt

def gauss_reduce(a, b, c):
    # Précondition (section 1) : a > 0, forme positive définie (D < 0).
    # Pour a <= 0 la descente n'a pas de sens : on refuse l'entrée.
    if a <= 0:
        raise ValueError('a doit être strictement positif (forme positive définie)')
    D = b * b - 4 * a * c
    while True:
        # Translation T^m de SL2(Z) : centrer b dans (-a, a] par pas de 2a.
        # C'est une translation (pas une réflexion) : la classe propre est préservée.
        b = b - ((b + a) // (2 * a)) * (2 * a)
        # c RECALCULÉ à chaque étape : seule façon de garder D = b^2 - 4ac invariant.
        c = (b * b - D) // (4 * a)
        if a > c:
            a, b, c = c, -b, a
            continue
        if (a == c or abs(b) == a) and b < 0:
            b = -b
        return (a, b, c)

print('Réduction de Gauss : invariance du discriminant et forme réduite')
print()
tests = [(1, 1, 1), (2, 1, 3), (5, 13, 9), (2, 5, 9), (9, 25, 21), (3, 3, 5), (1, 0, 5), (2, 2, 3), (7, 13, 9)]
print(f"{'Entrée':>13} | {'Réduite':>13} | {'D entrée':>9} | {'D sortie':>9} | invariant")
print('-' * 66)
for (a, b, c) in tests:
    D_in = b * b - 4 * a * c
    red = gauss_reduce(a, b, c)
    D_out = red[1] * red[1] - 4 * red[0] * red[2]
    ok = 'oui' if D_in == D_out else 'NON'
    print(f'{str((a, b, c)):>13} | {str(red):>13} | {D_in:>9} | {D_out:>9} | {ok}')

print()
print('Rejet de la précondition a > 0 :')
try:
    gauss_reduce(-3, 1, 1)
except ValueError as e:
    print(f'gauss_reduce(-3, 1, 1) -> ValueError : {e}')

Réduction de Gauss : invariance du discriminant et forme réduite

       Entrée |       Réduite |  D entrée |  D sortie | invariant
------------------------------------------------------------------
    (1, 1, 1) |     (1, 1, 1) |        -3 |        -3 | oui
    (2, 1, 3) |     (2, 1, 3) |       -23 |       -23 | oui
   (5, 13, 9) |     (1, 1, 3) |       -11 |       -11 | oui
    (2, 5, 9) |     (2, 1, 6) |       -47 |       -47 | oui
  (9, 25, 21) |     (5, 3, 7) |      -131 |      -131 | oui
    (3, 3, 5) |     (3, 3, 5) |       -51 |       -51 | oui
    (1, 0, 5) |     (1, 0, 5) |       -20 |       -20 | oui
    (2, 2, 3) |     (2, 2, 3) |       -20 |       -20 | oui
   (7, 13, 9) |     (3, 1, 7) |       -83 |       -83 | oui

Rejet de la précondition a > 0 :
gauss_reduce(-3, 1, 1) -> ValueError : a doit être strictement positif (forme positive définie)


### Lecture du résultat
Sur les 9 formes de test (toutes à $a > 0$, conformément à la précondition de la section 1), la sortie vérifie ligne à ligne les deux garanties de l'algorithme :
- **Invariance du discriminant** : la colonne « D sortie » est identique à « D entrée » à chaque ligne — chaque étape est une transformation $SL_2(\mathbb{Z})$ (translation $T^m$ pour centrer $b$ dans $(-a, a]$, puis swap $(a,b,c) \leftrightarrow (c,-b,a)$ tant que $a > c$), avec $c$ recalculé à chaque étape pour maintenir $D = b^2 - 4ac$.
- **Bornage de la forme réduite** : $|b| \leq a \leq c$, avec $b \geq 0$ dans les cas frontière $|b| = a$ ou $a = c$ — la sortie $(5, 3, 7)$ pour l'entrée $(9, 25, 21)$ en est l'illustration (trois itérations de swap).
La précondition $a > 0$ est enforced : une entrée à $a$ négatif est rejetée par un `ValueError` (dernière ligne de la sortie) — pour $a < 0$, la « réduction » $b \to b \pm 2a$ fait croître $|b|$ au lieu de le réduire, il n'y a pas de forme réduite dans cette direction.

## 2. Nombre de classes h(D) pour D < 0
Pour $D < 0$ discriminant quadratique fondamental, **h(D) = nombre de formes réduites** de discriminant D. C'est un algorithme en $O(\sqrt{|D|})$ : on itère sur $b \in [0, \sqrt{|D|/3}]$ (borne due à la condition $a \leq c \Rightarrow b^2 \leq -3D$), puis sur les diviseurs $a$ de $(b^2 - D)/4$, et on garde ceux qui satisfont $|b| \leq a \leq c$.
Pour les **9 corps quadratiques imaginaires de Heegner**, l'arithmétique exceptionnelle donne h(D) = 1. Cette liste est célèbre : c'est le théorème de Stark-Heegner (1952/1967), qui résout l'équation $x^2 + Dy^2 = $ premier en généralisant le théorème des deux carrés.

In [2]:
def enumerate_reduced_forms(D):
    forms = []
    b_step = 2
    b_parity = 0 if D % 4 == 0 else 1
    # Borne de réduction : |b| <= a <= sqrt(|D|/3) (cf. section 2) - la même
    # borne pour D = 0 mod 4 et D = 1 mod 4 (une borne -D/4 raterait des formes
    # comme (6, 6, 7) de D = -132).
    b_max = isqrt(-D // 3) if -D >= 3 else 0
    for b in range(b_parity, b_max + 1, b_step):
        if (b * b - D) % 4 != 0:
            continue
        a_c = (b * b - D) // 4
        for a in range(1, isqrt(a_c) + 1):
            if a_c % a != 0:
                continue
            c = a_c // a
            if abs(b) <= a <= c:
                if (a == c or abs(b) == a) and b < 0:
                    continue
                forms.append((a, b, c))
                # Miroir (a, -b, c) : classe PROPRE distincte quand
                # 0 < b < a < c (forme non ambiguë). L'omettre sous-compte h(D)
                # d'un facteur ~2 sur les discriminants génériques.
                if b > 0 and b < a and a < c:
                    forms.append((a, -b, c))
    return forms

HEEGNER_D = [-3, -4, -7, -8, -11, -19, -43, -67, -163]

print('Nombre de classes h(D) pour D < 0 (9 corps de Heegner + 4 controles a h>1)')
print()
print(f"{'D':>6} | {'h(D)':>4} | {'Formes réduites'}")
print('-' * 60)
test_D = HEEGNER_D + [-15, -20, -24, -35]
for D in test_D:
    forms = enumerate_reduced_forms(D)
    h = len(forms)
    if h <= 3:
        forms_str = str(forms)
    else:
        forms_str = f'({h} formes, premiere = {forms[0]})'
    print(f'{D:>5} | {h:>4} | {forms_str}')

Nombre de classes h(D) pour D < 0 (9 corps de Heegner + 4 controles a h>1)

     D | h(D) | Formes réduites
------------------------------------------------------------
   -3 |    1 | [(1, 1, 1)]
   -4 |    1 | [(1, 0, 1)]
   -7 |    1 | [(1, 1, 2)]
   -8 |    1 | [(1, 0, 2)]
  -11 |    1 | [(1, 1, 3)]
  -19 |    1 | [(1, 1, 5)]
  -43 |    1 | [(1, 1, 11)]
  -67 |    1 | [(1, 1, 17)]
 -163 |    1 | [(1, 1, 41)]
  -15 |    2 | [(1, 1, 4), (2, 1, 2)]
  -20 |    2 | [(1, 0, 5), (2, 2, 3)]
  -24 |    2 | [(1, 0, 6), (2, 0, 3)]
  -35 |    2 | [(1, 1, 9), (3, 1, 3)]


### Lecture du résultat
Les **9 corps quadratiques de Heegner** ont tous **h(D) = 1** :
- D = -3 : une seule forme $(1, 1, 1)$ (anneau des entiers de $\mathbb{Q}(\sqrt{-3})$ = $\mathbb{Z}[(1+\sqrt{-3})/2]$).
- D = -4 : forme $(1, 0, 1)$ (anneau des entiers de Gauss $\mathbb{Z}[i]$).
- D = -7 : forme $(1, 1, 2)$.
- D = -8 : forme $(1, 0, 2)$.
- D = -11 : forme $(1, 1, 3)$.
- D = -19 : forme $(1, 1, 5)$.
- D = -43 : forme $(1, 1, 11)$.
- D = -67 : forme $(1, 1, 17)$.
- D = -163 : forme $(1, 1, 41)$.
En control, les discriminants $-15, -20, -24, -35$ ont h(D) = 2, 2, 2, 2 respectivement (les plus petits non-Heegner). La liste de Heegner est **close** : c'est l'invariant h(D) = 1 qui définit le théorème de Stark-Heegner.

## 3. Vérification empirique du théorème de Stark-Heegner
Le **théorème de Stark-Heegner** (1952 pour h(D) ≤ 1, 1967 pour h(D) = 1 effectif) dit que la liste $\{D \in \mathbb{Z} : D \leq -3, h(D) = 1\}$ est exactement les 9 discriminants $\{-3, -4, -7, -8, -11, -19, -43, -67, -163\}$. C'est l'une des listes les plus célèbres de l'arithmétique.
La démonstration originale de Stark (1967) utilise l'analyse complexe (formule de Dirichlet pour h(D) comme residu de la $L$-série $L(1, \chi_D)$) ; celle de Heegner (1952) utilise les fonctions modulaires et les valeurs de la fonction $\eta$ de Dedekind.

In [3]:
print('Verification : h(D) = 1 sur tous les D entre -3 et -200 ?')
print()
HEEGNER_D = [-3, -4, -7, -8, -11, -19, -43, -67, -163]

def is_fundamental_disc(D):
    # Discriminant quadratique fondamental (Python : D % 4 in (0, 1) garde
    # les D négatifs congrus à 1 mod 4, que % == 1 exclusait à tort).
    if D % 4 not in (0, 1):
        return False
    if D % 4 == 1:
        m = D
    else:
        m = D // 4
        if m % 4 not in (2, 3):
            return False
    n = abs(m)
    for d in range(2, isqrt(n) + 1):
        if n % (d * d) == 0:
            return False
    return True

h_eq_1 = []
for D in range(-3, -200, -1):
    if not is_fundamental_disc(D):
        continue
    forms = enumerate_reduced_forms(D)
    if len(forms) == 1:
        h_eq_1.append(D)

print(f'Discriminants quadratiques fondamentaux D entre -3 et -200 avec h(D) = 1 :')
print(h_eq_1)
print()
print(f'Heegner connu : {HEEGNER_D[:9]}')
print()
matches = sorted(set(h_eq_1) & set(HEEGNER_D)) == sorted(h_eq_1)
print(f'Concordance numerique avec la liste de Heegner : {matches}')

Verification : h(D) = 1 sur tous les D entre -3 et -200 ?

Discriminants quadratiques fondamentaux D entre -3 et -200 avec h(D) = 1 :
[-3, -4, -7, -8, -11, -19, -43, -67, -163]

Heegner connu : [-3, -4, -7, -8, -11, -19, -43, -67, -163]

Concordance numerique avec la liste de Heegner : True


### Lecture du résultat
Avec le filtre de discriminant fondamental corrigé, l'algorithme de Gauss **retrouve** la liste de Heegner à partir de la seule condition h(D) = 1 sur l'intervalle D ∈ [-3, -200] :
- Pour chaque D discriminant quadratique fondamental (D ≡ 1 mod 4 avec |D| squarefree, ou D = 4m avec m ≡ 2, 3 mod 4 et |m| squarefree — le filtre exclut par exemple -12 = 4 × (-3) où -3 ≡ 1 mod 4), on énumère les formes réduites.
- La sortie donne les 9 discriminants h(D) = 1, exactement les Heegner : la concordance est **True**.
- Aucun autre D ∈ [-3, -200] fondamental n'a h(D) = 1.
C'est la **vérification numérique** du théorème de Stark-Heegner — pas la preuve (qui demande l'analyse complexe ou les fonctions modulaires), mais l'observation que la liste est close. Le carnet 07 (zéros de fonctions L) est lié : la formule analytique $h(D) = \frac{w \sqrt{|D|}}{2\pi} L(1, \chi_D)$ où $w$ est le nombre de racines de l'unité dans $\mathcal{O}_K$ (exercice 2 ci-dessous).

## 4. Exemples guidés

Deux calculs guidés, chacun dans une cellule **complète et exécutée** : la démarche est dans les commentaires, la sortie est commentée dans la cellule markdown qui suit.

1. le **balayage de h(D)** sur D ∈ [-500, -3], et le contrôle des strates h = 1, 2, 3 contre la liste de Heegner et la table des nombres de classes ;
2. la **formule analytique de Dirichlet** pour D = -163, recoupée par un produit d'Euler tronqué — deux voies indépendantes pour la même constante.

Les **exercices** du carnet sont en section 6 : ce sont des trames à compléter, distinctes de ces exemples résolus.


### Exemple guidé 1 — le balayage de h(D) et les petites valeurs

La section 3 ne testait que la condition h(D) = 1. On l'étend ici à tout l'intervalle D ∈ [-500, -3] : pour chaque discriminant fondamental, on énumère les formes réduites et on regroupe par valeur de h(D).

Le contrôle est **externe** : la liste des D à h(D) = 1 doit reproduire les 9 discriminants de Heegner, et les listes h(D) = 2 et h(D) = 3 se comparent à la table des nombres de classes (OEIS A014603 pour h = 1).


In [4]:
print('Exo 1 : h(D) pour D entre -3 et -500, lister h=2 et h=3')
print()
h1_list = []
h2_list = []
h3_list = []
for D in range(-500, -2):
    if not is_fundamental_disc(D):
        continue
    forms = enumerate_reduced_forms(D)
    h = len(forms)
    if h == 1:
        h1_list.append(D)
    elif h == 2:
        h2_list.append(D)
    elif h == 3:
        h3_list.append(D)

print(f'h(D) = 1 ({len(h1_list)} discriminants) : {sorted(h1_list)}')
print(f'Concordance Heegner : {sorted(h1_list) == sorted(HEEGNER_D)}')
print()
print(f'h(D) = 2 ({len(h2_list)} discriminants) : {sorted(h2_list)}')
print()
print(f'h(D) = 3 ({len(h3_list)} discriminants) : {sorted(h3_list)}')

Exo 1 : h(D) pour D entre -3 et -500, lister h=2 et h=3

h(D) = 1 (9 discriminants) : [-163, -67, -43, -19, -11, -8, -7, -4, -3]
Concordance Heegner : True

h(D) = 2 (18 discriminants) : [-427, -403, -267, -235, -232, -187, -148, -123, -115, -91, -88, -52, -51, -40, -35, -24, -20, -15]

h(D) = 3 (12 discriminants) : [-499, -379, -331, -307, -283, -211, -139, -107, -83, -59, -31, -23]


### Lecture du résultat

Le balayage sur 500 unités de discriminant retrouve exactement les 9 discriminants de Heegner pour h(D) = 1 — `Concordance Heegner : True` — là où la section 3 s'arrêtait à l'intervalle [-200, -3].

Les deux strates suivantes donnent le spectre des petites valeurs de h :

- **h(D) = 2 : 18 discriminants**, de -15 à -427 ;
- **h(D) = 3 : 12 discriminants**, de -23 (la plus petite valeur de h = 3) à -499.

Les listes s'espacent quand |D| croît : c'est la croissance moyenne de h(D), de l'ordre de √|D| à un facteur logarithmique près, qui rend chaque strate de plus en plus rare. La liste h = 1, elle, est **close** — les 9 valeurs sont toutes celles qui existent, c'est le théorème de Stark-Heegner (section 3).


### Exemple guidé 2 — la formule analytique du nombre de classes

La formule de Dirichlet relie l'invariant **arithmétique** h(D) à une valeur de fonction L :

$$h(D) = \frac{w\sqrt{|D|}}{2\pi}\,L(1, \chi_D),$$

où `w` est le nombre de racines de l'unité du corps (w = 2 pour D < -4) et $\chi_D$ le caractère quadratique associé.

Pour D = -163 (le plus grand discriminant de Heegner), la cellule calcule les deux membres **par des voies indépendantes** : h(D) par énumération des formes réduites, puis $L(1, \chi_D)$ par un produit d'Euler tronqué $\prod_{p \le N} (1 - \chi_D(p)/p)^{-1}$, évalué à N = 10⁴ puis N = 10⁵. L'accord entre les deux membres est le contrôle.


In [5]:
import math

print('Exo 2 : formule analytique de Dirichlet pour D = -163')
print()
D = -163
h = len(enumerate_reduced_forms(D))
w = 2  # unités ±1 (D < -4)
sqrtD = math.sqrt(-D)

# Côté 1 : formule de classes de Dirichlet, avec h compté par énumération des formes
L1_dirichlet = 2 * math.pi * h / (w * sqrtD)
print(f'h(-163) = {h}, w = {w} : L(1, chi_D) = 2*pi*h/(w*sqrt(|D|)) = {L1_dirichlet:.6f}')
print()

# chi_D(p) = (D/p) : critère d'Euler pour p impair, cas p = 2 à part
# (chi_D(2) = +1 si D = ±1 mod 8, -1 si D = ±3 mod 8).
def euler_product_L1(D, N):
    sieve = bytearray([1]) * (N + 1)
    sieve[0:2] = b'\x00\x00'
    prod = 1.0
    if D % 2 != 0:
        chi2 = 1 if D % 8 in (1, 7) else -1
        prod /= (1 - chi2 / 2)
    for p in range(3, N + 1, 2):
        if not sieve[p]:
            continue
        for m in range(p * p, N + 1, p):
            sieve[m] = 0
        if D % p == 0:
            continue  # premier ramifié : facteur local 1
        chi = pow(D, (p - 1) // 2, p)
        prod /= (1 - (1 if chi == 1 else -1) / p)
    return prod

L1_euler = None
for N in (10**4, 10**5):
    L1_euler = euler_product_L1(D, N)
    écart = abs(L1_euler - L1_dirichlet)
    print(f'N = {N:>6} : L(1, chi_D) par produit d Euler = {L1_euler:.6f}, écart = {écart:.2e}')
print()
print(f'Concordance (écart < 1e-3 à N = 10^5) : {abs(L1_euler - L1_dirichlet) < 1e-3}')

Exo 2 : formule analytique de Dirichlet pour D = -163

h(-163) = 1, w = 2 : L(1, chi_D) = 2*pi*h/(w*sqrt(|D|)) = 0.246069

N =  10000 : L(1, chi_D) par produit d Euler = 0.245458, écart = 6.10e-04


N = 100000 : L(1, chi_D) par produit d Euler = 0.245861, écart = 2.07e-04

Concordance (écart < 1e-3 à N = 10^5) : True


### Lecture du résultat

Les deux voies donnent la même constante :

- la **formule de Dirichlet** donne $L(1, \chi_D) = 2\pi h / (w\sqrt{|D|}) = 0.246069$ pour h(-163) = 1 et w = 2 ;
- le **produit d'Euler tronqué** donne `0.245458` à N = 10⁴ puis `0.245861` à N = 10⁵, soit des écarts de `6.10e-04` puis `2.07e-04`.

L'écart décroît d'un facteur ≈ 3 quand N est multiplié par 10 : c'est la convergence lente attendue d'un produit d'Euler au bord de sa région de convergence (le terme d'erreur se comporte comme $1/\log N$, non comme $1/N$). La dernière ligne teste l'accord avec la tolérance 1e-3 à N = 10⁵.


## 5. La composition de Gauss

Jusqu'ici le carnet a **compté** les classes. Gauss va plus loin : il les **multiplie**. Deux formes de même discriminant se composent en une troisième, et cette opération fait de l'ensemble des classes un **groupe abélien fini** d'ordre h(D) — le groupe des classes Cl(D), dont l'ordre est précisément le nombre de classes calculé en section 2.

**Pourquoi une multiplication ?** Par la correspondance de la section 1, une forme (a, b, c) de discriminant D < 0 représente l'idéal $I = a\mathbb{Z} + \frac{b + \sqrt{D}}{2}\mathbb{Z}$ de l'ordre de discriminant D. Composer deux classes revient à **multiplier leurs idéaux** : le produit $I_1 I_2$ admet lui aussi une base réduite, et c'est elle qui définit la forme composée. La classe de $I_1 I_2$ ne dépend que des classes de $I_1$ et de $I_2$ — c'est ce qui rend la loi bien définie sur les classes.

**L'algorithme (Lagrange ; Gauss, *Disquisitiones* art. 234-243).** Pour $f_1 = (a_1, b_1, c_1)$ et $f_2 = (a_2, b_2, c_2)$ de même discriminant D :

1. $e = \mathrm{pgcd}(a_1, a_2, (b_1 + b_2)/2)$ — le « contenu » commun aux deux formes (section 5.1) ;
2. $A = a_1 a_2 / e^2$ ;
3. $B$ est déterminé modulo $2A$ par les congruences $B \equiv b_1 \ (\mathrm{mod}\ 2a_1/e)$, $B \equiv b_2 \ (\mathrm{mod}\ 2a_2/e)$ et $B^2 \equiv D \ (\mathrm{mod}\ 4A)$ ;
4. la forme $(A, B, (B^2 - D)/(4A))$ est la composée ; on la **réduit** par l'algorithme de la section 1 pour en obtenir le représentant canonique.

**Coût, mesuré sur l'implémentation.** L'algorithme ci-dessus, résolu par le théorème des restes chinois, serait polynomial en $\log \max(a_1, a_2, |D|)$. **Ce carnet ne résout pas les congruences** : la cellule 5.2 **balaie** les candidats $B$ de $0$ à $2A$ jusqu'à ce que les trois congruences soient satisfaites — un balayage **linéaire en $A$**, pas polylogarithmique. Sur des formes réduites ($a \leq \sqrt{|D|/3}$, donc $A \leq |D|/3$), il reste **borné** : c'est un choix pédagogique — voir les congruences agir une à une — et il ne faut pas lui attribuer la complexité d'une implémentation efficace par congruences.


### 5.1 Le pgcd `e`, contenu commun des deux formes

Quand `e = 1`, les deux formes sont dites **unies** (*united*) : la composition est alors donnée directement par les formules ci-dessus. L'étape `e` mesure le facteur commun aux deux formes — c'est elle qui explique le $e^2$ au dénominateur de A : le produit des idéaux contient une copie redondante du contenu, qu'il faut retirer.

La cellule ci-dessous isole cette première étape sur trois couples de formes de discriminant -7, -11 et -23.


In [6]:
from math import gcd

def gauss_compose_step1(a1, b1, c1, a2, b2, c2):
    if (b1 + b2) % 2 != 0:
        raise ValueError('b1 + b2 doit etre pair')
    D = b1*b1 - 4*a1*c1
    if D != b2*b2 - 4*a2*c2:
        raise ValueError('Discriminants incompatibles')
    g = gcd(gcd(a1, a2), (b1 + b2) // 2)
    return g

print('Section 5.1 : le pgcd e = pgcd(a1, a2, (b1 + b2)/2), contenu commun aux deux formes')
print()
tests = [((1, 1, 2), (1, 1, 2), -7), ((1, 1, 3), (1, 1, 3), -11), ((2, 1, 3), (2, 1, 3), -23)]
for (a1, b1, c1), (a2, b2, c2), D in tests:
    g = gauss_compose_step1(a1, b1, c1, a2, b2, c2)
    print(f'gcd pour compose({(a1,b1,c1)}, {(a2,b2,c2)}) avec D={D} : g = {g}')

Section 5.1 : le pgcd e = pgcd(a1, a2, (b1 + b2)/2), contenu commun aux deux formes

gcd pour compose((1, 1, 2), (1, 1, 2)) avec D=-7 : g = 1
gcd pour compose((1, 1, 3), (1, 1, 3)) avec D=-11 : g = 1
gcd pour compose((2, 1, 3), (2, 1, 3)) avec D=-23 : g = 1


### Lecture du résultat

Les trois couples testés sont **unis** : `e = 1` dans les trois cas (D = -7, -11 et -23). Le choix est délibéré — deux exemplaires de la même forme n'ont aucun contenu commun à partager.

Le cas intéressant est `e > 1` : il apparaît dès que l'on compose deux formes **distinctes** d'un même discriminant. Par exemple `(2, 1, 3)` avec `(2, -1, 3)` pour D = -23 : les deux coefficients `b` s'annulent ($b_1 + b_2 = 0$), et `e = pgcd(2, 2, 0) = 2`. C'est là que le facteur $e^2$ travaille. La section 5.2 le mesure sur les tables de multiplication complètes.


### 5.2 La composition complète, et la loi de groupe

La cellule ci-dessous implémente l'algorithme complet, puis **vérifie les axiomes de groupe** sur tous les discriminants fondamentaux de [-250, -3] — soit, à chaque discriminant, la table de multiplication entière des h(D) classes :

- **fermeture** : le composé de deux formes réduites est encore une forme réduite de la liste ;
- **élément neutre** : la forme principale, $(1, 0, -D/4)$ pour $D \equiv 0 \ (\mathrm{mod}\ 4)$ et $(1, 1, (1-D)/4)$ pour $D \equiv 1 \ (\mathrm{mod}\ 4)$ ;
- **inverse** : $(a, -b, c)$ est l'inverse de $(a, b, c)$ — la forme conjuguée ;
- **commutativité** : $f \cdot g = g \cdot f$ ;
- **associativité** : $(f \cdot g) \cdot h = f \cdot (g \cdot h)$.

Elle confronte ensuite le nombre d'éléments d'ordre divisant 2 à la **théorie du genre** de Gauss : le 2-rang de Cl(D) vaut $\omega(D) - 1$, où $\omega(D)$ est le nombre de facteurs premiers distincts de |D|. C'est le pont avec le carnet 11, qui avait mesuré la **parité** de h(D) par le même invariant.


In [7]:
# Composition de Gauss : la loi de groupe sur les classes (section 5).
# Correspondance forme <-> ideal (section 5) : une forme (a, b, c) de discriminant
# D < 0 represente l'ideal a*Z + ((b + sqrt(D))/2)*Z ; composer deux classes revient
# a multiplier leurs ideaux. L'algorithme ci-dessous est celui de Lagrange
# (Gauss, Disquisitiones art. 234-243) : pgcd -> A -> congruences sur B -> reduction.
from math import gcd


def forme_principale(D):
    # Classe neutre du groupe : la forme de discriminant D dont le coefficient a
    # vaut 1. Elle represente l'anneau tout entier, donc la classe triviale.
    if D % 4 == 0:
        return (1, 0, -D // 4)
    return (1, 1, (1 - D) // 4)


def gauss_compose(f1, f2):
    a1, b1, c1 = f1
    a2, b2, c2 = f2
    D = b1 * b1 - 4 * a1 * c1
    if D >= 0:
        raise ValueError('composition implementee pour D < 0 (formes definies positives)')
    if b2 * b2 - 4 * a2 * c2 != D:
        raise ValueError('les deux formes doivent avoir le meme discriminant')
    e = gcd(gcd(a1, a2), (b1 + b2) // 2)
    A = (a1 * a2) // (e * e)
    # B est determine modulo 2A par les congruences B = b1 (mod 2a1/e),
    # B = b2 (mod 2a2/e) et B^2 = D (mod 4A) : on balaie le domaine [0, 2A].
    for B in range(2 * A + 1):
        if (B - b1) % (2 * a1 // e) != 0 or (B - b2) % (2 * a2 // e) != 0:
            continue
        if (B * B - D) % (4 * A) != 0:
            continue
        return gauss_reduce(A, B, (B * B - D) // (4 * A))
    raise ValueError('aucun B : formes non primitives ou discriminants incompatibles')


def classe(f):
    # Representant canonique d'une classe : la forme reduite de Gauss (section 1).
    return gauss_reduce(*f)


print('Composition de Gauss : le produit de deux classes (section 5)')
print()
D_demo = -23
neutre = classe(forme_principale(D_demo))
f_demo = (2, 1, 3)
f2 = classe(gauss_compose(f_demo, f_demo))
f3 = classe(gauss_compose(f2, f_demo))
print(f'D = {D_demo} : h(D) = {len(enumerate_reduced_forms(D_demo))}, classe neutre = {neutre}')
print(f'  f     = {f_demo}')
print(f'  f^2   = {f2}   <- f x f')
print(f'  f^3   = {f3}   <- retour a la classe neutre : f est d ordre 3')
print(f'  f x neutre = {classe(gauss_compose(f_demo, neutre))}   (inchange)')
print(f'  f x conjuguee (2, -1, 3) = {classe(gauss_compose(f_demo, (2, -1, 3)))}   (neutre)')
print()


# ---------------------------------------------------------------------------
# Verification des axiomes sur les discriminants fondamentaux de [-250, -3] :
# table de multiplication complete, puis 2-torsion contre la theorie du genre.
# ---------------------------------------------------------------------------
def facteurs_premiers(n):
    fs, p = set(), 2
    while p * p <= n:
        if n % p == 0:
            fs.add(p)
            while n % p == 0:
                n //= p
        p += 1
    if n > 1:
        fs.add(n)
    return fs


DETAIL = [-3, -4, -7, -8, -11, -15, -19, -20, -23, -24, -35, -40, -47, -71, -84, -120, -163]

print('Axiomes de groupe et theorie du genre (section 5.2)')
print()
print(f"{'D':>6} | {'h':>3} | {'ferm':>5} | {'neutre':>6} | {'inverse':>7} | {'abelien':>7} | {'assoc':>5} | {'2-torsion':>9} | {'2^(w-1)':>7}")
print('-' * 86)

axiomes_ok = 0
genre_ok = 0
nb_disc = 0
paires = 0
paires_e_sup_1 = 0
for D in range(-3, -250, -1):
    if not is_fundamental_disc(D):
        continue
    forms = enumerate_reduced_forms(D)
    if not forms:
        continue
    nb_disc += 1
    neutre = classe(forme_principale(D))
    ferm = neut = inv = abel = assoc = True
    for x in forms:
        if classe(gauss_compose(x, neutre)) != x or classe(gauss_compose(neutre, x)) != x:
            neut = False
        # L'inverse d'une classe est la classe de la forme conjuguee (a, -b, c).
        if classe(gauss_compose(x, classe((x[0], -x[1], x[2])))) != neutre:
            inv = False
        for y in forms:
            paires += 1
            if gcd(gcd(x[0], y[0]), (x[1] + y[1]) // 2) > 1:
                paires_e_sup_1 += 1
            xy = classe(gauss_compose(x, y))
            if xy not in forms:
                ferm = False
            if xy != classe(gauss_compose(y, x)):
                abel = False
            for z in forms:
                if classe(gauss_compose(xy, z)) != classe(gauss_compose(x, classe(gauss_compose(y, z)))):
                    assoc = False
    torsion = sum(1 for x in forms if classe(gauss_compose(x, x)) == neutre)
    omega = len(facteurs_premiers(abs(D)))
    predit = 2 ** (omega - 1)
    if torsion == predit:
        genre_ok += 1
    if ferm and neut and inv and abel and assoc:
        axiomes_ok += 1
    if D in DETAIL:
        print(f'{D:>6} | {len(forms):>3} | {str(ferm):>5} | {str(neut):>6} | {str(inv):>7} | {str(abel):>7} | {str(assoc):>5} | {torsion:>9} | {predit:>7}')

print('-' * 86)
print()
print(f'Axiomes de groupe verifies : {axiomes_ok}/{nb_disc} discriminants fondamentaux de [-250, -3]')
print(f'Compositions calculees : {paires} paires, dont {paires_e_sup_1} a e > 1 (le facteur e^2 est actif)')
print(f'2-torsion = 2^(omega(|D|) - 1) : {genre_ok}/{nb_disc} accords')


Composition de Gauss : le produit de deux classes (section 5)

D = -23 : h(D) = 3, classe neutre = (1, 1, 6)
  f     = (2, 1, 3)
  f^2   = (2, -1, 3)   <- f x f
  f^3   = (1, 1, 6)   <- retour a la classe neutre : f est d ordre 3
  f x neutre = (2, 1, 3)   (inchange)
  f x conjuguee (2, -1, 3) = (1, 1, 6)   (neutre)

Axiomes de groupe et theorie du genre (section 5.2)

     D |   h |  ferm | neutre | inverse | abelien | assoc | 2-torsion | 2^(w-1)
--------------------------------------------------------------------------------------
    -3 |   1 |  True |   True |    True |    True |  True |         1 |       1
    -4 |   1 |  True |   True |    True |    True |  True |         1 |       1
    -7 |   1 |  True |   True |    True |    True |  True |         1 |       1
    -8 |   1 |  True |   True |    True |    True |  True |         1 |       1
   -11 |   1 |  True |   True |    True |    True |  True |         1 |       1
   -15 |   2 |  True |   True |    True |    True |  True |  

  -120 |   4 |  True |   True |    True |    True |  True |         4 |       4
  -163 |   1 |  True |   True |    True |    True |  True |         1 |       1


--------------------------------------------------------------------------------------



Axiomes de groupe verifies : 77/77 discriminants fondamentaux de [-250, -3]
Compositions calculees : 2502 paires, dont 632 a e > 1 (le facteur e^2 est actif)
2-torsion = 2^(omega(|D|) - 1) : 77/77 accords


### Lecture du résultat

La sortie centrale du carnet tient en trois lignes, chacune une conclusion :

- **D = -23, h(D) = 3** : la forme f = (2, 1, 3) engendre tout le groupe — f² = (2, -1, 3), puis f³ = neutre = (1, 1, 6). Cl(-23) est **cyclique d'ordre 3**, et f × conjuguée redonne le neutre : l'inverse est bien la classe conjuguée.
- **77/77 discriminants fondamentaux de [-250, -3]** : fermeture, neutre, inverse, commutativité et associativité sont vérifiées partout où elles ont été testées — la table ci-dessus n'en montre que 17, le compte final porte sur les 77.
- **2502 paires composées, dont 632 avec e > 1** : le facteur e² de la section 5.1 n'est pas une rareté théorique — un quart des compositions le mobilisent, et la loi reste bien définie.

Le pont avec la **théorie du genre** de Gauss tient aussi : sur les 77 discriminants, le nombre d'éléments d'ordre divisant 2 vaut toujours $2^{\omega(|D|)-1}$, où $\omega$ compte les facteurs premiers distincts de |D| (77/77 accords). C'est le même invariant qui mesurait la parité de h(D) dans le carnet 11.

**Limite de ces vérifications** : elles sont **empiriques et bornées**. Elles portent sur les formes **primitives, définies positives**, des discriminants fondamentaux jusqu'à -250. Aucun de ces 77 accords n'est une preuve générale — ni de la loi de groupe, ni de la formule du genre : ils disent seulement que sur ce domaine, rien ne contredit Gauss.


## 6. Exercices

Trois exercices à compléter. Chacun est une cellule **trame** : elle s'exécute telle quelle (aucune erreur n'est levée), mais sa sortie ne mesure rien tant que le corps n'est pas écrit — les valeurs attendues sont affichées à côté pour servir de contrôle.

Les trois s'appuient sur les outils construits plus haut : `enumerate_reduced_forms` (section 2), puis `gauss_compose`, `forme_principale` et `classe` (section 5).


### Exercice 1 — cyclique, ou produit de cycliques ?

Deux groupes abéliens finis de même ordre peuvent être non isomorphes : pour h(D) = 4, le groupe des classes peut être cyclique ($\mathbb{Z}/4$) ou un produit de deux groupes d'ordre 2 ($\mathbb{Z}/2 \times \mathbb{Z}/2$). La théorie du genre tranche : le 2-rang vaut $\omega(D) - 1$, donc dès que |D| a trois facteurs premiers distincts, le groupe n'est pas cyclique.

**Objectif** : écrire `structure_groupe(D)` qui renvoie `'Z/n'` quand le groupe est cyclique d'ordre n, et `'Z/2 x Z/2 x ...'` quand il est un **produit de groupes d'ordre 2** — en calculant l'**ordre de chaque élément**.

**Contrat borné** : les cinq discriminants de contrôle ci-dessous, dont les groupes sont chacun cycliques ou de Klein. Un groupe abélien fini quelconque peut mêler les deux familles ($D = -231$ porte un facteur d'ordre 3 à côté de son 2-rang $\omega(D) - 1 = 2$) ; sa décomposition complète en produit de cycliques dépasse le cadre de cet exercice.

**Indice** : l'ordre du groupe est `len(enumerate_reduced_forms(D))` ; un groupe d'ordre n est cyclique s'il contient un élément d'ordre n.

**Étape 1** : écrire `ordre(g, D)` — plus petit k ≥ 1 tel que la puissance k-ième de g soit la classe neutre.
**Étape 2** : en déduire `structure_groupe(D)` (ordre maximal = n ⟹ cyclique).
**Étape 3** : contrôler sur les cinq discriminants de la cellule.


In [8]:
# Exercice 1 : structure du groupe des classes (cyclique, ou produit de Z/2)
def ordre(g, D):
    # TODO etudiant : plus petit k >= 1 tel que la puissance k-ieme de g soit la classe neutre.
    # Etape 1 : neutre = classe(forme_principale(D)) ; partir de y = classe(g), k = 1.
    # Etape 2 : tant que y != neutre, faire y = classe(gauss_compose(y, g)) et k += 1.
    return None


def structure_groupe(D):
    # TODO etudiant : renvoyer 'Z/n' si le groupe est cyclique d'ordre n (un element
    # d'ordre n existe), 'Z/2 x Z/2 x ...' s'il est un produit de groupes d'ordre 2.
    # Contrat borne aux cinq discriminants de controles ci-dessous (cycliques ou Klein) ;
    # un groupe abelien quelconque (ex. D = -231) peut combiner les deux familles a la fois.
    # Indice : ordre maximal = len(enumerate_reduced_forms(D)) <=> cyclique.
    return None


controles = [(-84, 'Z/2 x Z/2'), (-47, 'Z/5'), (-23, 'Z/3'), (-120, 'Z/2 x Z/2'), (-71, 'Z/7')]
for D, attendu in controles:
    print(f'D = {D:>5} : h = {len(enumerate_reduced_forms(D))}, structure = {structure_groupe(D)} '
          f'(attendu : {attendu})')


D =   -84 : h = 4, structure = None (attendu : Z/2 x Z/2)
D =   -47 : h = 5, structure = None (attendu : Z/5)
D =   -23 : h = 3, structure = None (attendu : Z/3)
D =  -120 : h = 4, structure = None (attendu : Z/2 x Z/2)
D =   -71 : h = 7, structure = None (attendu : Z/7)


### Exercice 2 — Cl(-71) est-il cyclique ?

Le discriminant D = -71 a h(D) = 7, et 7 est premier : les seuls groupes d'ordre 7 sont cycliques. La question intéressante n'est donc pas **si** Cl(-71) est cyclique, mais **quel élément** l'engendre, et à quelle vitesse.

**Objectif** : pour la forme `f = (2, 1, 9)` de D = -71, calculer la suite des puissances successives jusqu'au retour à la classe neutre, puis vérifier que les 7 puissances parcourent **tout** le groupe.

**Indice** : `gauss_compose` renvoie une forme quelconque de la classe ; il faut la **réduire** (`classe`) avant toute comparaison à la forme neutre, sinon deux représentants de la même classe se comparent comme différents.

**Étape 1** : construire la liste des puissances successives de `f`.
**Étape 2** : s'arrêter quand la classe neutre réapparaît, **en l'incluant dans la liste** — la liste finale compte exactement `ordre` éléments, neutre compris ; c'est ce que l'étape 3 attend pour couvrir les 7 classes de Cl(-71).
**Étape 3** : vérifier que l'ensemble des puissances égale `enumerate_reduced_forms(-71)`.


In [9]:
# Exercice 2 : l'ordre de la classe de (2, 1, 9) dans Cl(-71)
f71 = (2, 1, 9)
D71 = -71


def ordre_classe(f, D):
    # TODO etudiant : renvoyer (ordre, liste des puissances reduites) jusqu'au retour au neutre.
    # Etape 1 : neutre = classe(forme_principale(D)).
    # Etape 2 : y = classe(f) ; accumuler y dans une liste, en posant
    #           y = classe(gauss_compose(y, f)) a chaque tour, et arreter EN INCLUANT
    #           le retour au neutre : la liste finale compte exactement `ordre`
    #           elements, neutre compris (six puissances non neutres ne suffisent
    #           pas -- le controle de l'etape 3 attend les 7 classes de Cl(-71)).
    return None, None


ordre, puissances = ordre_classe(f71, D71)
print(f'f = {f71}, D = {D71} : ordre = {ordre} (attendu : 7)')
couvre = puissances is not None and sorted(set(puissances)) == sorted(enumerate_reduced_forms(D71))
print(f'les puissances parcourent-elles tout le groupe ? {couvre}')


f = (2, 1, 9), D = -71 : ordre = None (attendu : 7)
les puissances parcourent-elles tout le groupe ? False


### Exercice 3 — le sous-groupe des carrés Cl(D)²

L'application $x \mapsto x^2$ est un endomorphisme du groupe des classes : son image Cl(D)² est un sous-groupe. Comme |G| = |ker| · |im| pour un groupe fini, et que le noyau est l'ensemble des éléments d'ordre divisant 2, on a

$$|Cl(D)^2| = \frac{h(D)}{2^{\omega(D) - 1}}.$$

**Objectif** : vérifier cette relation, puis en déduire le critère : Cl(D)² = Cl(D) si et seulement si h(D) est **impair**.

**Indice** : `h(D) = len(enumerate_reduced_forms(D))` ; $\omega(D)$ se calcule en factorisant |D| — `facteurs_premiers` de la section 5.2 est réutilisable.

**Étape 1** : construire l'ensemble $\{x^2 : x \in Cl(D)\}$.
**Étape 2** : comparer son cardinal à $h(D)/2^{\omega(D)-1}$.
**Étape 3** : balayer les discriminants fondamentaux de [-200, -3] et vérifier l'équivalence « carrés = groupe entier ⟺ h impair ».


In [10]:
# Exercice 3 : le sous-groupe des carres Cl(D)^2
def carres(D):
    # TODO etudiant : renvoyer l'ensemble des classes x^2, x parcourant Cl(D).
    # Etape 1 : forms = enumerate_reduced_forms(D).
    # Etape 2 : {classe(gauss_compose(x, x)) for x in forms}.
    return None


def omega(D):
    # TODO etudiant : nombre de facteurs premiers distincts de |D|.
    # Indice : facteurs_premiers (section 5.2) est reutilisable.
    return None


for D in (-15, -23, -47, -84, -120):
    h = len(enumerate_reduced_forms(D))
    n_carres = None if carres(D) is None else len(carres(D))
    print(f'D = {D:>5} : h = {h:>2}, |Cl^2| = {n_carres} (attendu h / 2^(w-1))')

accords = None
print(f'Balayage [-200, -3] : equivalence "carres = groupe entier" <-> "h impair" : {accords}')


D =   -15 : h =  2, |Cl^2| = None (attendu h / 2^(w-1))
D =   -23 : h =  3, |Cl^2| = None (attendu h / 2^(w-1))
D =   -47 : h =  5, |Cl^2| = None (attendu h / 2^(w-1))
D =   -84 : h =  4, |Cl^2| = None (attendu h / 2^(w-1))
D =  -120 : h =  4, |Cl^2| = None (attendu h / 2^(w-1))
Balayage [-200, -3] : equivalence "carres = groupe entier" <-> "h impair" : None


## Conclusion
Le **nombre de classes** h(D) est l'invariant arithmétique fondamental d'un corps quadratique. Pour D < 0, c'est un entier calculable par énumération des formes réduites (Gauss). Pour D > 0, h(D) est **fini lui aussi** — c'est le **groupe des unités** qui est infini (±ε^k, théorème des unités de Dirichlet) ; les formes réduites indéfinies s'y lisent en cycles, un par classe.
La liste des 9 corps de Heegner (h(D) = 1) est close et vérifiable : c'est le théorème de Stark-Heegner, à la croisée de l'arithmétique, de l'analyse et de la théorie des nombres.
L'ensemble des classes n'est donc pas un simple compteur : c'est un **groupe abélien fini**, dont l'ordre est h(D). C'est la lecture moderne du calcul de Gauss.
Suite naturelle :
- La **composition de Gauss** est désormais livrée (section 5) : `gauss_compose` implémente l'algorithme de Lagrange, et les axiomes de groupe — fermeture, élément neutre, inverse, commutativité, associativité — sont vérifiés sur les discriminants fondamentaux de [-250, -3].
- Piste ouverte : le **corps de classes de Hilbert** — l'extension abélienne maximale non ramifiée de Q(√D) a pour groupe de Galois Cl(D). Relier h(D) à un degré d'extension est le point de départ de la théorie des corps de classes.
- Pont avec la série GenAI : la formule analytique $h(D) = \frac{w \sqrt{|D|}}{2\pi} L(1, \chi_D)$ est reliée à l'étude des zéros de fonctions L (carnet 07).

## Ressources
- **Serre**, *A Course in Arithmetic*, chapitre 4 (Springer GTM 7) — référence canonique.
- **Cox**, *Primes of the Form x² + ny²* (Wiley) — chapitre 7 sur la composition de Gauss.
- **Gauss**, *Disquisitiones Arithmeticae* §§ 222-256 (1801) — l'original.
- **Stark** (1967), *On the "gap" in a theorem of Heegner* — la preuve d'effectivité.
- **Heegner** (1952), *Diophantische Analysis und Modulfunktionen* — la construction.
- **Buell**, *Binary Quadratic Forms: Classical Theory and Modern Computations* (Springer) — la composition des formes, du pgcd à la forme réduite.
- **Cohen**, *A Course in Computational Algebraic Number Theory* (Springer GTM 138) — composition des formes et réduction, dans la forme algorithmique utilisée ici.
- OEIS A014603 — discriminants quadratiques imaginaires fondamentaux avec h(D) = 1.